# Used Car Sales & Pricing Analytics

**AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026**

**Project objective:** Analyze a used-car dataset to understand pricing patterns, fuel/transmission preferences, seller types, ownership, mileage, vehicle age, and the relationship between these factors and selling price.

> **Dataset:** Vehicle dataset from CarDekho, available on Kaggle.  
> **Important:** The notebook is designed to run with the downloaded CSV placed in the same folder as this notebook. It also contains an optional URL fallback for environments with internet access.


## 1. Business Problem

Used-car businesses and buyers need to understand what factors are associated with resale prices. This project uses descriptive analytics and basic statistical/modeling techniques to answer:

1. How are used-car selling prices distributed?
2. Which fuel types, seller types and transmissions are most common?
3. How does vehicle age relate to selling price?
4. How does mileage relate to selling price?
5. Which manufacturers/brands appear most frequently?
6. How does average selling price differ across categories?
7. Can a simple regression model estimate selling price from the available attributes?

**Scope:** This is an exploratory analytics project. Relationships found in observational listing data should not be interpreted as proof of causation.


In [ ]:
# 2. Install/import libraries
# If running locally, install dependencies with: pip install -r requirements.txt

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


In [ ]:
# 3. Load the dataset
# Preferred method: download the Kaggle CSV and place it beside this notebook.
# Expected filename: CAR DETAILS FROM CAR DEKHO.csv

LOCAL_FILES = [
    "CAR DETAILS FROM CAR DEKHO.csv",
    "car_data.csv",
    "CAR DETAILS FROM CAR DEKHO - CAR DETAILS FROM CAR DEKHO.csv"
]

df = None
for f in LOCAL_FILES:
    if os.path.exists(f):
        df = pd.read_csv(f)
        print(f"Loaded local file: {f}")
        break

# Optional fallback: public GitHub mirror of the same 8-column dataset.
if df is None:
    url = "https://raw.githubusercontent.com/bagassenop/cardekho/main/CAR%20DETAILS%20FROM%20CAR%20DEKHO.csv"
    try:
        df = pd.read_csv(url)
        print("Loaded dataset from the optional public mirror.")
    except Exception as e:
        raise FileNotFoundError(
            "Dataset not found. Download the Kaggle CSV and place it beside this notebook "
            "with the filename 'CAR DETAILS FROM CAR DEKHO.csv'."
        ) from e

df.head()


In [ ]:
# 4. Initial inspection
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing"))

print("\nDuplicate rows:", df.duplicated().sum())
display(df.head(10))


## 2. Data Cleaning & Preparation

The dataset contains vehicle name, year, selling price, kilometers driven, fuel, seller type, transmission and owner information. We standardize text fields, remove exact duplicate rows, validate numeric fields and create derived variables such as vehicle age and brand.


In [ ]:
# 5. Clean and standardize data
data = df.copy()

text_cols = ["name", "fuel", "seller_type", "transmission", "owner"]
for c in text_cols:
    data[c] = data[c].astype(str).str.strip()

for c in ["year", "selling_price", "km_driven"]:
    data[c] = pd.to_numeric(data[c], errors="coerce")

before = len(data)
data = data.drop_duplicates().copy()
duplicates_removed = before - len(data)

# Remove rows with invalid core numeric values
data = data.dropna(subset=["year", "selling_price", "km_driven"]).copy()
data = data[(data["selling_price"] > 0) & (data["km_driven"] >= 0)].copy()

# Extract brand from the first word of the car name.
# This is a simple rule; multi-word brands may need manual standardization.
data["brand"] = data["name"].str.split().str[0].str.title()

# Vehicle age relative to the latest year represented in the dataset
reference_year = int(data["year"].max())
data["vehicle_age"] = reference_year - data["year"]

print("Rows before cleaning:", before)
print("Exact duplicates removed:", duplicates_removed)
print("Rows after cleaning:", len(data))
print("Reference year used for vehicle age:", reference_year)

display(data.head())


In [ ]:
# 6. Descriptive statistics
display(data[["year", "selling_price", "km_driven", "vehicle_age"]].describe().T)

print("\nUnique values by categorical field:")
for c in ["fuel", "seller_type", "transmission", "owner", "brand"]:
    print(f"{c}: {data[c].nunique()}")


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# 7. Selling price distribution
plt.figure(figsize=(10, 5))
sns.histplot(data["selling_price"], bins=40, kde=True)
plt.title("Distribution of Used-Car Selling Prices")
plt.xlabel("Selling Price (INR)")
plt.ylabel("Number of Cars")
plt.tight_layout()
plt.show()


In [ ]:
# 8. Fuel type distribution
fuel_counts = data["fuel"].value_counts()

plt.figure(figsize=(9, 5))
sns.barplot(x=fuel_counts.index, y=fuel_counts.values)
plt.title("Cars by Fuel Type")
plt.xlabel("Fuel Type")
plt.ylabel("Number of Cars")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

display(fuel_counts.to_frame("count"))


In [ ]:
# 9. Transmission and seller type
fig, ax = plt.subplots(figsize=(8, 5))
trans_counts = data["transmission"].value_counts()
sns.barplot(x=trans_counts.index, y=trans_counts.values, ax=ax)
ax.set_title("Cars by Transmission")
ax.set_xlabel("Transmission")
ax.set_ylabel("Number of Cars")
plt.tight_layout()
plt.show()

display(data["seller_type"].value_counts().to_frame("count"))


In [ ]:
# 10. Top 15 brands by number of listings
top_brands = data["brand"].value_counts().head(15)

plt.figure(figsize=(10, 6))
sns.barplot(y=top_brands.index, x=top_brands.values)
plt.title("Top 15 Brands by Number of Listings")
plt.xlabel("Number of Listings")
plt.ylabel("Brand")
plt.tight_layout()
plt.show()

display(top_brands.to_frame("listings"))


In [ ]:
# 11. Average selling price by fuel type
avg_fuel_price = data.groupby("fuel")["selling_price"].agg(["count", "mean", "median"]).sort_values("mean", ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=avg_fuel_price.index, y=avg_fuel_price["mean"])
plt.title("Average Selling Price by Fuel Type")
plt.xlabel("Fuel Type")
plt.ylabel("Average Selling Price (INR)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

display(avg_fuel_price)


In [ ]:
# 12. Average selling price by transmission
avg_trans_price = data.groupby("transmission")["selling_price"].agg(["count", "mean", "median"]).sort_values("mean", ascending=False)
display(avg_trans_price)

plt.figure(figsize=(8, 5))
sns.boxplot(data=data, x="transmission", y="selling_price")
plt.title("Selling Price by Transmission")
plt.xlabel("Transmission")
plt.ylabel("Selling Price (INR)")
plt.ylim(0, data["selling_price"].quantile(0.99))
plt.tight_layout()
plt.show()


In [ ]:
# 13. Mileage vs selling price
plt.figure(figsize=(10, 6))
sns.scatterplot(data=data, x="km_driven", y="selling_price", alpha=0.5)
plt.title("Mileage vs Selling Price")
plt.xlabel("Kilometers Driven")
plt.ylabel("Selling Price (INR)")
plt.ylim(0, data["selling_price"].quantile(0.99))
plt.tight_layout()
plt.show()

print("Correlation (km_driven vs selling_price):",
      data["km_driven"].corr(data["selling_price"]))


In [ ]:
# 14. Vehicle age vs selling price
plt.figure(figsize=(10, 6))
sns.scatterplot(data=data, x="vehicle_age", y="selling_price", alpha=0.5)
sns.regplot(data=data, x="vehicle_age", y="selling_price", scatter=False)
plt.title("Vehicle Age vs Selling Price")
plt.xlabel("Vehicle Age (years)")
plt.ylabel("Selling Price (INR)")
plt.ylim(0, data["selling_price"].quantile(0.99))
plt.tight_layout()
plt.show()

print("Correlation (vehicle_age vs selling_price):",
      data["vehicle_age"].corr(data["selling_price"]))


In [ ]:
# 15. Average price by owner category
owner_price = data.groupby("owner")["selling_price"].agg(["count", "mean", "median"]).sort_values("mean", ascending=False)
display(owner_price)

plt.figure(figsize=(10, 5))
sns.barplot(x=owner_price.index, y=owner_price["mean"])
plt.title("Average Selling Price by Ownership Category")
plt.xlabel("Owner Category")
plt.ylabel("Average Selling Price (INR)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


## 4. Statistical Relationship / Correlation

Correlation is used here as a descriptive measure of linear association. A correlation does **not** establish causation.


In [ ]:
# 16. Correlation matrix for numerical variables
num_cols = ["year", "selling_price", "km_driven", "vehicle_age"]
corr = data[num_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

display(corr)


## 5. Basic Predictive Model

A simple linear regression model is included as an additional analytical exercise. Categorical fields are one-hot encoded. The model is intended to demonstrate the workflow, not to provide a production pricing engine.


In [ ]:
# 17. Prepare features for a simple regression model
model_df = data[[
    "year", "km_driven", "fuel", "seller_type", "transmission", "owner", "selling_price"
]].copy()

X = model_df.drop(columns="selling_price")
y = model_df["selling_price"]

categorical_features = ["fuel", "seller_type", "transmission", "owner"]
numeric_features = ["year", "km_driven"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)

print(f"MAE : ₹{mae:,.0f}")
print(f"RMSE: ₹{rmse:,.0f}")
print(f"R²  : {r2:.3f}")


In [ ]:
# 18. Actual vs predicted prices
results = pd.DataFrame({"Actual": y_test.values, "Predicted": pred})

plt.figure(figsize=(8, 6))
sns.scatterplot(data=results, x="Actual", y="Predicted", alpha=0.5)
min_v = min(results["Actual"].min(), results["Predicted"].min())
max_v = max(results["Actual"].max(), results["Predicted"].max())
plt.plot([min_v, max_v], [min_v, max_v], linestyle="--")
plt.title("Actual vs Predicted Selling Price")
plt.xlabel("Actual Price (INR)")
plt.ylabel("Predicted Price (INR)")
plt.tight_layout()
plt.show()


## 6. Key Findings Generator

Run this cell after executing the analysis cells. It prints dataset-specific findings so the final report can be updated with the exact numbers produced by your run.


In [ ]:
# 19. Automatically generate concise findings
most_common_fuel = data["fuel"].mode().iat[0]
most_common_trans = data["transmission"].mode().iat[0]
most_common_seller = data["seller_type"].mode().iat[0]
top_brand = data["brand"].value_counts().idxmax()

print("KEY FINDINGS")
print("-" * 60)
print(f"1. Most common fuel type: {most_common_fuel}")
print(f"2. Most common transmission: {most_common_trans}")
print(f"3. Most common seller type: {most_common_seller}")
print(f"4. Most frequent brand: {top_brand}")
print(f"5. Median selling price: ₹{data['selling_price'].median():,.0f}")
print(f"6. Median kilometers driven: {data['km_driven'].median():,.0f}")
print(f"7. Correlation of mileage with price: {data['km_driven'].corr(data['selling_price']):.3f}")
print(f"8. Correlation of vehicle age with price: {data['vehicle_age'].corr(data['selling_price']):.3f}")
print(f"9. Regression R² on the held-out test set: {r2:.3f}")


## 7. Conclusion

The analysis combines data cleaning, exploratory analysis, visualization, correlation analysis and a basic predictive model. The final interpretation should focus on observed patterns in this dataset rather than treating them as universal rules for every used-car market.

### Suggested business takeaways
- Segment pricing analysis by vehicle age, mileage, fuel type and transmission.
- Avoid using average price alone; median and distribution plots are useful because vehicle prices are unevenly distributed.
- Treat extreme high-price observations carefully when communicating typical market prices.
- Use the model as a baseline and add stronger vehicle attributes before using it for operational pricing.


## 8. Dataset & Source Notes

Primary dataset page:
https://www.kaggle.com/nehalbirla/vehicle-dataset-from-cardekho

Optional public mirror used only as a runtime fallback:
https://github.com/bagassenop/cardekho

The Kaggle dataset is described as used-car data with eight fields: name, year, selling_price, km_driven, fuel, seller_type, transmission and owner. The Kaggle page identifies the dataset as an Open Database / Database Contents resource.

**Submission note:** Keep the downloaded CSV in the same folder as the notebook before running all cells.
